# Fine-tune NLLB-200 for English → Idoma

**Run this on Colab with a T4 (free tier is enough).** `Runtime → Change runtime type → T4 GPU`.

---

## Why this notebook exists

The deployed app returned English instead of Idoma. The cause is not a bug in the
serving code — it is that **stock NLLB-200 does not support Idoma at all**. Its
tokenizer holds exactly 202 language codes, and `idu_Latn` is not one of them:

```
eng_Latn -> 256047   valid
ibo_Latn -> 256073   valid   (Igbo)
idu_Latn -> 3        <unk>   (Idoma — absent)
ig_Latn  -> 3        <unk>   (not a real NLLB code either)
```

`generate(forced_bos_token_id=<unk>)` gives the decoder no target-language signal,
so it copies the source sentence. Reproduced directly:

```
INPUT: Come and eat
  idu_Latn (id=3)      -> 'Come and eat'      <-- the reported bug
  ibo_Latn (id=256073) -> 'Bịa rie nri .'     <-- a valid code translates fine
```

So the fix is a checkpoint that genuinely has an Idoma token. This notebook:

1. adds `idu_Latn` to the tokenizer as a new special token,
2. resizes the model's embedding matrix to fit it,
3. **initialises the new embedding row from `ibo_Latn`** — Igbo is the nearest
   in-vocabulary Benue-Congo relative, so the new token starts somewhere sensible
   instead of at random,
4. fine-tunes on the corpus from `data_pipeline/`,
5. asserts `idu_Latn != <unk>` before saving (the guard whose absence caused the bug),
6. scores chrF++ on the held-out test split,
7. pushes to **your own public, ungated** repo.

## Expectations, honestly

The open corpus is **1,251 pairs over 1,117 distinct English headwords** — measured,
not estimated: 1,117 of idomaland.org's 1,119 dictionary pages parsed (99.82%), split
989 train / 125 dev / 137 test. Mostly word-level. That is enough to
make the model produce real Idoma vocabulary; it is **not** enough for fluent
sentence translation. Keep the dictionary-lookup-first ordering in the backend so
exact hits stay exact. If access to `mrheartng/idoma-english-parallel-corpus`
(10K+) comes through, re-run this notebook with it added and quality jumps.

## 1. Install

In [ ]:
!pip install -q "transformers>=4.40" "datasets>=2.18" "accelerate>=0.29" \
                sentencepiece sacrebleu evaluate huggingface_hub

import torch, transformers
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
assert torch.cuda.is_available(), "Enable the GPU: Runtime -> Change runtime type -> T4"

## 2. Configuration

Set `HUB_REPO` to your own namespace. Leave it public and ungated — the whole
reason the original deployment fell back to a broken model is that the reference
checkpoint was gated.

In [ ]:
BASE_MODEL   = "facebook/nllb-200-distilled-600M"

SRC_LANG     = "eng_Latn"
TGT_LANG     = "idu_Latn"   # Idoma, ISO 639-3 'idu'. Added by this notebook.
SEED_LANG    = "ibo_Latn"   # Igbo: nearest in-vocabulary relative, used to init.

HUB_REPO     = "YOUR_HF_USERNAME/nllb-eng-idoma"  # <-- CHANGE ME
PUSH_TO_HUB  = True

OUTPUT_DIR   = "/content/nllb-eng-idoma"

# Small corpus -> more epochs, low LR, and both directions so the encoder sees
# Idoma text too.
EPOCHS            = 12
LEARNING_RATE     = 3e-5
BATCH_SIZE        = 8
GRAD_ACCUM        = 2
MAX_LENGTH        = 96
LABEL_SMOOTHING   = 0.1
TRAIN_BOTH_DIRECTIONS = True

## 3. Log in to the Hub

Needs a **write** token: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login, whoami

login()  # paste a WRITE token
print("logged in as:", whoami()["name"])

## 4. Upload the corpus

Build it locally first:

```bash
python3 data_pipeline/scrape_idomaland.py --delay 8
python3 data_pipeline/build_corpus.py
```

then upload `data_pipeline/out/train.jsonl`, `dev.jsonl`, and `test.jsonl` here.
Each line is `{"en": ..., "idu": ..., "dialect": ..., "source": ..., "url": ...}`.

The corpus is **not** published — it stays on your machine and in this runtime.

In [ ]:
import os, json, pathlib

DATA_DIR = pathlib.Path("/content/data")
DATA_DIR.mkdir(exist_ok=True, parents=True)

if not (DATA_DIR / "train.jsonl").exists():
    try:
        from google.colab import files
        print("Select train.jsonl, dev.jsonl and test.jsonl ...")
        for name, blob in files.upload().items():
            (DATA_DIR / name).write_bytes(blob)
    except ImportError:
        raise SystemExit(f"Not on Colab — copy the .jsonl files into {DATA_DIR}")

for name in ("train.jsonl", "dev.jsonl", "test.jsonl"):
    path = DATA_DIR / name
    count = sum(1 for _ in path.open(encoding="utf-8")) if path.exists() else 0
    print(f"{name}: {count} rows")

assert (DATA_DIR / "train.jsonl").exists(), "train.jsonl is required"

### Sanity-check the data before spending GPU time

`build_corpus.py` already enforces these, but a corpus that echoes its input or
carries placeholders will silently produce a model with the original symptom.
Check again here — it costs a second.

In [ ]:
def read_jsonl(path):
    if not pathlib.Path(path).exists():
        return []
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

train_rows = read_jsonl(DATA_DIR / "train.jsonl")
dev_rows   = read_jsonl(DATA_DIR / "dev.jsonl")
test_rows  = read_jsonl(DATA_DIR / "test.jsonl")

PLACEHOLDER = "\u1ee5\u1ee5"  # the fabricated filler from the old dictionary

bad_placeholder = [r for r in train_rows if PLACEHOLDER in r["idu"]]
bad_identical   = [r for r in train_rows if r["en"].strip().lower() == r["idu"].strip().lower()]
assert not bad_placeholder, f"{len(bad_placeholder)} placeholder rows in train"
assert not bad_identical,   f"{len(bad_identical)} echo rows in train (these cause the copy bug)"

# Splits are keyed on the English side, so no headword should straddle them.
train_keys = {r["en"].lower() for r in train_rows}
leaked = train_keys & {r["en"].lower() for r in test_rows}
print(f"train={len(train_rows)} dev={len(dev_rows)} test={len(test_rows)}")
print(f"train/test English overlap: {len(leaked)}", "OK" if not leaked else f"LEAK: {sorted(leaked)[:10]}")

if len(train_rows) < 500:
    print(f"\nWARNING: only {len(train_rows)} training rows. Expect vocabulary-level")
    print("competence, not fluent sentences. Add the gated corpus if you get access.")

for row in train_rows[:5]:
    print(" ", row["en"], "->", row["idu"], f"({row.get('dialect') or 'unspecified'})")

## 5. Load the base model and confirm the bug

This cell reproduces the failure before fixing it, so there is no doubt about the
diagnosis.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, src_lang=SRC_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

unk = tokenizer.unk_token_id
for code in (SRC_LANG, SEED_LANG, TGT_LANG, "ig_Latn"):
    tid = tokenizer.convert_tokens_to_ids(code)
    print(f"{code:10s} -> {tid:<8} {'<unk> MISSING' if tid == unk else 'ok'}")

print(f"\nNLLB language codes in this tokenizer: "
      f"{sum(1 for t in tokenizer.additional_special_tokens if '_' in t)}")
assert tokenizer.convert_tokens_to_ids(TGT_LANG) == unk, (
    f"{TGT_LANG} already exists — you are not starting from stock NLLB")

In [ ]:
# Demonstrate the copy behaviour: forcing <unk> as the first decoder token.
device = "cuda"
model.to(device).eval()
probe = "Come and eat"
tokenizer.src_lang = SRC_LANG
inputs = tokenizer(probe, return_tensors="pt").to(device)

for code in (TGT_LANG, SEED_LANG):
    tid = tokenizer.convert_tokens_to_ids(code)
    with torch.no_grad():
        out = model.generate(**inputs, forced_bos_token_id=tid, max_length=64, num_beams=4)
    print(f"{code} (id={tid}) -> {tokenizer.decode(out[0], skip_special_tokens=True)!r}")
print(f"\nINPUT was {probe!r} — note that the <unk> row returns it unchanged.")

## 6. THE FIX: add `idu_Latn`, resize, and seed it from `ibo_Latn`

This is the step whose absence caused the whole problem.

In [ ]:
seed_id = tokenizer.convert_tokens_to_ids(SEED_LANG)
assert seed_id != tokenizer.unk_token_id, f"{SEED_LANG} must exist to seed from"

# 1. Register the new language code as a special token.
added = tokenizer.add_special_tokens(
    {"additional_special_tokens": tokenizer.additional_special_tokens + [TGT_LANG]}
)
print(f"tokens added: {added}")

new_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
assert new_id != tokenizer.unk_token_id, "failed to add the token"
print(f"{TGT_LANG} -> {new_id}")

# 2. Grow the embedding matrix to cover the new id.
old_size = model.get_input_embeddings().weight.shape[0]
model.resize_token_embeddings(len(tokenizer))
new_size = model.get_input_embeddings().weight.shape[0]
print(f"embeddings: {old_size} -> {new_size}")

# 3. Copy Igbo's embedding into the new row. A randomly initialised language token
#    on ~1k training rows barely moves; starting from a related Benue-Congo
#    language gives the decoder a usable prior from step one.
with torch.no_grad():
    embeddings = model.get_input_embeddings().weight
    embeddings[new_id] = embeddings[seed_id].clone()

    output_embeddings = model.get_output_embeddings()
    if output_embeddings is not None and not model.config.tie_word_embeddings:
        output_embeddings.weight[new_id] = output_embeddings.weight[seed_id].clone()

# Keep the generation config coherent with the new target language.
model.config.forced_bos_token_id = new_id
if getattr(model, "generation_config", None) is not None:
    model.generation_config.forced_bos_token_id = new_id

# NLLB tokenizers keep their own code->id map; keep it in sync so
# `tokenizer.src_lang = 'idu_Latn'` works for the reverse direction.
for attr in ("lang_code_to_id", "_lang_code_to_id"):
    mapping = getattr(tokenizer, attr, None)
    if isinstance(mapping, dict):
        mapping[TGT_LANG] = new_id
        print(f"updated tokenizer.{attr}")

print("\nfix applied")

## 7. Tokenise

`text_target=` makes the tokenizer emit labels prefixed with the *target* language
token, which is what teaches the model what `idu_Latn` means.

In [ ]:
from datasets import Dataset

def to_pairs(rows, both_directions):
    """Expand rows into directional training examples."""
    pairs = [{"src": r["en"], "tgt": r["idu"], "src_lang": SRC_LANG, "tgt_lang": TGT_LANG}
             for r in rows]
    if both_directions:
        pairs += [{"src": r["idu"], "tgt": r["en"], "src_lang": TGT_LANG, "tgt_lang": SRC_LANG}
                  for r in rows]
    return pairs

train_pairs = to_pairs(train_rows, TRAIN_BOTH_DIRECTIONS)
dev_pairs   = to_pairs(dev_rows, False)   # eval English -> Idoma only
print(f"train examples: {len(train_pairs)} | dev examples: {len(dev_pairs)}")

def tokenize(batch):
    # One tokenizer instance carries a single src_lang, so group by direction.
    input_ids, attention, labels = [], [], []
    for src, tgt, sl, tl in zip(batch["src"], batch["tgt"],
                                batch["src_lang"], batch["tgt_lang"]):
        tokenizer.src_lang = sl
        tokenizer.tgt_lang = tl
        encoded = tokenizer(src, text_target=tgt, truncation=True, max_length=MAX_LENGTH)
        input_ids.append(encoded["input_ids"])
        attention.append(encoded["attention_mask"])
        labels.append(encoded["labels"])
    return {"input_ids": input_ids, "attention_mask": attention, "labels": labels}

train_ds = Dataset.from_list(train_pairs).map(
    tokenize, batched=True, remove_columns=["src", "tgt", "src_lang", "tgt_lang"])
dev_ds = Dataset.from_list(dev_pairs).map(
    tokenize, batched=True, remove_columns=["src", "tgt", "src_lang", "tgt_lang"]) \
    if dev_pairs else None

# Verify the labels really start with the Idoma token.
sample = train_ds[0]["labels"]
print("first label token:", tokenizer.convert_ids_to_tokens([sample[0]]))
print("decoded label:", tokenizer.decode(sample, skip_special_tokens=True))

## 8. Train

In [ ]:
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=0.1,
    weight_decay=0.01,
    label_smoothing_factor=LABEL_SMOOTHING,
    fp16=True,
    logging_steps=25,
    eval_strategy="epoch" if dev_ds else "no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=bool(dev_ds),
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    report_to=[],
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collator,
    processing_class=tokenizer,
)

trainer.train()

## 9. The guard that must pass before deploying

If either assertion below fails, the model will exhibit the original bug. Do not
push it.

In [ ]:
def translate(texts, src_lang=SRC_LANG, tgt_lang=TGT_LANG, num_beams=4):
    tokenizer.src_lang = src_lang
    tid = tokenizer.convert_tokens_to_ids(tgt_lang)
    assert tid != tokenizer.unk_token_id, f"{tgt_lang} resolves to <unk>"
    batch = tokenizer(texts, return_tensors="pt", padding=True,
                      truncation=True, max_length=MAX_LENGTH).to(model.device)
    model.eval()
    with torch.no_grad():
        out = model.generate(**batch, forced_bos_token_id=tid,
                             max_length=MAX_LENGTH, num_beams=num_beams)
    return tokenizer.batch_decode(out, skip_special_tokens=True)


# GUARD 1: the language token exists.
tid = tokenizer.convert_tokens_to_ids(TGT_LANG)
assert tid != tokenizer.unk_token_id, f"FAIL: {TGT_LANG} is still <unk>"
print(f"GUARD 1 ok: {TGT_LANG} -> {tid}")

# GUARD 2: the model does not just echo its input.
probes = ["water", "Come and eat", "good morning", "my father", "How are you?"]
outputs = translate(probes)
echoed = [(p, o) for p, o in zip(probes, outputs) if p.strip().lower() == o.strip().lower()]
for probe, output in zip(probes, outputs):
    flag = "ECHO" if probe.strip().lower() == output.strip().lower() else "ok"
    print(f"  [{flag}] {probe!r} -> {output!r}")
assert len(echoed) < len(probes), "FAIL: the model echoes every input — the original bug"
print(f"\nGUARD 2 ok: {len(probes) - len(echoed)}/{len(probes)} probes produced new text")

print("\nReverse direction (Idoma -> English):")
for src, out in zip(["Ennkpo", "Le"], translate(["Ennkpo", "Le"], TGT_LANG, SRC_LANG)):
    print(f"  {src!r} -> {out!r}")

## 10. Score chrF++ on the held-out test split

chrF++ rather than BLEU: the test set is mostly short word-level pairs, and BLEU's
4-gram precision is close to meaningless on those. chrF++ works on characters, so
it also gives partial credit for near-miss diacritics.

In [ ]:
import sacrebleu
from collections import defaultdict

if test_rows:
    # Group references by English so multiple dialect variants all count as correct.
    by_source = defaultdict(list)
    for row in test_rows:
        by_source[row["en"]].append(row["idu"])

    sources = list(by_source)
    hypotheses = []
    for start in range(0, len(sources), 16):
        hypotheses.extend(translate(sources[start:start + 16]))

    max_refs = max(len(v) for v in by_source.values())
    references = [[(by_source[s][i] if i < len(by_source[s]) else by_source[s][0])
                   for s in sources] for i in range(max_refs)]

    chrf = sacrebleu.corpus_chrf(hypotheses, references, word_order=2)
    bleu = sacrebleu.corpus_bleu(hypotheses, references)
    exact = sum(h.strip().lower() in [r.lower() for r in by_source[s]]
                for s, h in zip(sources, hypotheses))
    echoes = sum(s.strip().lower() == h.strip().lower() for s, h in zip(sources, hypotheses))

    print(f"test sources : {len(sources)}")
    print(f"chrF++       : {chrf.score:.2f}")
    print(f"BLEU         : {bleu.score:.2f}  (noisy on word-level data)")
    print(f"exact match  : {exact}/{len(sources)} ({100 * exact / len(sources):.1f}%)")
    print(f"echoed input : {echoes}/{len(sources)}  <-- must be near zero")

    print("\nsamples:")
    for s, h in list(zip(sources, hypotheses))[:15]:
        print(f"  {s!r} -> {h!r}   (ref: {by_source[s]})")
else:
    print("no test.jsonl — skipping evaluation")

## 11. Save and push

Public and ungated, so the Space can actually load it.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Round-trip check: reloading must still resolve the Idoma token. If the token did
# not survive serialisation, the served model reverts to the original bug.
reloaded_tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
rid = reloaded_tok.convert_tokens_to_ids(TGT_LANG)
assert rid != reloaded_tok.unk_token_id, f"{TGT_LANG} did not survive save/load"
print(f"reload ok: {TGT_LANG} -> {rid}")

In [ ]:
MODEL_CARD = f"""---
license: cc-by-nc-4.0
language:
- en
- idu
base_model: {BASE_MODEL}
pipeline_tag: translation
tags:
- translation
- nllb
- idoma
- low-resource
---

# NLLB-200 fine-tuned for English <-> Idoma

Idoma (ISO 639-3 `idu`, Glottolog `idom1241`) is a Benue-Congo language of Benue
State, Nigeria. **Stock NLLB-200 does not support it** — its tokenizer has 202
language codes and `idu_Latn` is not among them, so `idu_Latn` resolves to `<unk>`
and the model returns the source sentence unchanged.

This checkpoint adds `idu_Latn` as a real token, resizes the embedding matrix, and
initialises the new row from `ibo_Latn` (Igbo, the nearest in-vocabulary relative)
before fine-tuning.

## Usage

```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained("{HUB_REPO}")
model = AutoModelForSeq2SeqLM.from_pretrained("{HUB_REPO}")

# This must NOT be the unk id.
tgt = tok.convert_tokens_to_ids("idu_Latn")
assert tgt != tok.unk_token_id

tok.src_lang = "eng_Latn"
batch = tok("Come and eat", return_tensors="pt")
out = model.generate(**batch, forced_bos_token_id=tgt, max_length=96, num_beams=4)
print(tok.decode(out[0], skip_special_tokens=True))
```

## Training data

Roughly {len(train_rows)} English-Idoma pairs, predominantly word-level, derived
from the community dictionary at <https://www.idomaland.org/dictionary> with
gratitude to its contributors. The dataset itself is not redistributed here.

Entries are dialect-tagged where the source distinguishes them (central vs western
Idoma differ: *water* is **Ennkpo** in central and **Enyi** in western).

## Limitations

- Trained on a small, mostly word-level corpus: treat it as a
  **dictionary-augmented translator**, not a fluent sentence translator.
- Tone marking is inconsistent in the source data. Tone is contrastive in Idoma
  (*àkpà* "bridge" vs *ákpá* "cloud"), so some outputs will be tonally wrong.
- Dialect coverage is uneven; central Idoma dominates.
- Do not use for anything safety-critical.

## Evaluation

chrF++ on a held-out split, keyed by English headword so dialect variants of the
same word cannot leak between train and test. See the training notebook
(`training/train_idoma_nllb.ipynb`) for exact numbers from your run.
"""

with open(f"{OUTPUT_DIR}/README.md", "w", encoding="utf-8") as handle:
    handle.write(MODEL_CARD)
print(MODEL_CARD[:900])

In [ ]:
if PUSH_TO_HUB:
    assert "YOUR_HF_USERNAME" not in HUB_REPO, "Set HUB_REPO to your own namespace first"
    from huggingface_hub import HfApi

    api = HfApi()
    # private=False and no gating: a gated model is what broke the deployment.
    api.create_repo(HUB_REPO, repo_type="model", private=False, exist_ok=True)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HUB_REPO, repo_type="model")
    print(f"pushed -> https://huggingface.co/{HUB_REPO}")
    print("\nVerify it is ungated by fetching a file without a token:")
    print(f"  curl -sI https://huggingface.co/{HUB_REPO}/resolve/main/config.json | head -1")
    print("  (expect 200 — a 401 means the repo is gated and the app will fall back)")
else:
    print("PUSH_TO_HUB is False — nothing uploaded")

## 12. Wire it into the app

Set this on both the Hugging Face Space and the backend host:

```bash
NMT_MODEL_ID=<your HUB_REPO>
```

`translator_service/config.py` reads it, and `app.py` / `services/nmt_service.py`
check at load time that `idu_Latn` is present — if you point them at a checkpoint
without it, they now report a configuration error instead of quietly returning
English.

Then confirm end to end:

```bash
curl -s localhost:5005/translate \
  -H 'Content-Type: application/json' \
  -d '{"text":"water","source_lang":"English","target_lang":"Idoma"}'
# expect Ennkpo / Enyi — not "water"
```